# Parser v7 — RTL annotation with the Atlas fixes

Runs **Algorithm 1 line 4**: mechanical extraction decides *what exists*, an LLM decides
*what it means*. Writes `parsed_v7_tuning18/`.

This is a self-contained copy. It does not read or write anything in `Prepoison_subset/`,
so the v3 parse and everything measured on it stay exactly as they are.

## What changed from v3, and why

Each change traces to a measurement in the Parser Failure Atlas (`docs/`), taken from the
verifier run over all 1 689 elements.

| # | change | the measurement behind it |
|---|---|---|
| 1 | `SEQUENCES` narrowed to the clock; **`RESETS` added** | the v3 prompt said "clock **or reset edge**", so resets were labelled `SEQUENCES` — **111 claims, 31.8% of every `SEQUENCES` claim**, contradicted by the RTL |
| 2 | **`guard` required** on `GATES`/`SELECTS`/`CONSTRAINS`/`OVERRIDES` | governing coverage **51%**, and **48.2%** of the `GATES` edges that did exist were incomplete. There was nowhere to record *under what condition* |
| 3 | `EXPORTS` restricted to outputs | "reaches the entity boundary" never said which way — **27 input ports** were tagged `EXPORTS` |
| 4 | `ISOLATED` checked mechanically, in code | the prompt already forbids it on a used element and the parser did it anyway — the signature of a rule that belongs in code |

**Reset was deliberately NOT folded into `OVERRIDES`.** `OVERRIDES` means a *bypass*, and
bypass paths are the U-category assets. There are 11 `OVERRIDES` entries in the whole v3
parse against ~76 reset relationships; pushing reset in would make reset 87% of that bucket
and destroy a signal to fix a different one.

**The self-gating case needs no new rule.** A lock bit appears as `pmpcfg GATES pmpcfg`,
which reads as a meaningless self-loop. Change 2 fixes it for free — the bit number is part
of the guard being quoted. Requiring `bits` only for self-edges would have invited the model
to add `bits` everywhere.

## 1. Configuration

In [1]:
# RUN FROM THE FOLDER ROOT, not from notebooks/.
# prompts_parse_v3.audit_against_corpus() is called with no arguments inside
# annotate_all_parallel(), so it resolves ground_truth/ and parsed_tuning18/ relative to the
# working directory. Both live at the folder root.
import os, sys, json, re, hashlib
from pathlib import Path
from collections import Counter

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
print("cwd:", Path.cwd())

RTL_DIR   = Path("data/RTL_data")        # 18 .vhd files
V3_DIR    = Path("data/parsed_v3_tuning18")   # the previous parse, for comparison only
V7_DIR    = Path("parsed_v7_tuning18")   # this run's output
REF_DIR   = Path("parsed_tuning18")      # v1 closed set -- the coverage reference
GT_FILE   = Path("ground_truth/manual_gt_neorv32.json")

GEN_MODEL     = "gpt-5-mini"
PARSE_WORKERS = 6
PARSE_EFFORT  = "medium"

for p in (RTL_DIR, REF_DIR, GT_FILE):
    print(f"  {str(p):32s} {'OK' if p.exists() else 'MISSING'}")

cwd: e:\jobs\ff\test\Prepoison_subset\bahavioral_patterns_of_assets
  data\RTL_data                    OK
  parsed_tuning18                  OK
  ground_truth\manual_gt_neorv32.json OK


In [2]:
# ---- API key -----------------------------------------------------------------
def load_env_file(path="API.env"):
    p = Path(path); loaded = []
    if not p.exists(): return loaded
    for raw in p.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line: continue
        k, v = line.split("=", 1)
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if k: os.environ[k] = v; loaded.append(k)
    return loaded

print("loaded from API.env:", load_env_file() or "nothing")
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing -- put it in API.env"
print(f"OPENAI_API_KEY: present ({len(os.environ['OPENAI_API_KEY'])} chars)")

loaded from API.env: ['OPENAI_API_KEY']
OPENAI_API_KEY: present (164 chars)


In [3]:
# ---- the client. Same shape as finetuning_assetgen_v2.ipynb cell 3 ------------
import time, random
from openai import OpenAI

class LLMClient:
    def __init__(self, model=GEN_MODEL, effort=PARSE_EFFORT, max_retries=6):
        self.c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model, self.effort, self.r = model, effort, max_retries
        self.usage = []

    def _call(self, system, user, json_mode, max_tokens, effort):
        kw = dict(model=self.model, instructions=system, input=user,
                  reasoning={"effort": effort or self.effort},
                  max_output_tokens=max_tokens)
        if json_mode:
            kw["text"] = {"format": {"type": "json_object"}}
        wait = 2.0
        for a in range(self.r):
            try:
                r = self.c.responses.create(**kw)
                if getattr(r, "status", None) == "incomplete":
                    why = getattr(getattr(r, "incomplete_details", None), "reason", "?")
                    print(f"    [llm warn] incomplete ({why}) at max_output_tokens={max_tokens}")
                u = r.usage
                self.usage.append({"in": u.input_tokens, "out": u.output_tokens,
                                   "reasoning": u.output_tokens_details.reasoning_tokens,
                                   "cached_in": getattr(u.input_tokens_details, "cached_tokens", 0)})
                return r.output_text
            except Exception as e:
                if a == self.r - 1: raise
                time.sleep(wait + random.uniform(0, 1)); wait *= 2

    def complete(self, system, user, max_tokens=4096, effort=None):
        return self._call(system, user, True, max_tokens, effort)

    def cost(self, price_in=0.25, price_out=2.00, price_cached=0.025):
        i = sum(u["in"] - u["cached_in"] for u in self.usage)
        c = sum(u["cached_in"] for u in self.usage)
        o = sum(u["out"] for u in self.usage)
        return {"calls": len(self.usage), "in": i, "cached_in": c, "out": o,
                "usd": round((i*price_in + c*price_cached + o*price_out)/1e6, 4)}

client = LLMClient()
print("client ready:", GEN_MODEL, "| effort", PARSE_EFFORT)

client ready: gpt-5-mini | effort medium


## 2. The closed set

`rtl_parse` extracts every port and signal mechanically. The model never adds or drops an
element — `verify_against_cache` raises if the annotated set differs from the mechanical one
by a single entry, so a coverage failure is a hard error rather than a judgement call.

`RECORDS` is **required**. Without it the record-typed ports declared in the package do not
expand, and the closed set silently differs from the one every other stage uses.

In [4]:
import importlib, rtl_parse, prompts_parse_v3, parse_v3
for m in (rtl_parse, prompts_parse_v3, parse_v3):
    importlib.reload(m)

# point the driver at THIS folder's directories
parse_v3.OUT_DIR   = V7_DIR
parse_v3.CACHE_DIR = REF_DIR

modules = [(p.stem, p) for p in sorted(RTL_DIR.glob("*.vhd"))]
assert len(modules) == 18, f"expected 18 modules, found {len(modules)}"
RECORDS = rtl_parse.build_record_registry(sorted(RTL_DIR.glob("*.vhd")))
print(f"{len(modules)} modules | {len(RECORDS)} record types "
      f"(bus_req_t={'bus_req_t' in RECORDS}, bus_rsp_t={'bus_rsp_t' in RECORDS})")

# consistency: the mechanical set must equal the v1 reference, element for element
ok = 0
for stem, path in modules:
    mech = rtl_parse.parse_rtl_file(path, records=RECORDS)
    ref = json.loads((REF_DIR / f"{stem}.json").read_text(encoding="utf-8"))
    gp = [(e["entity"], x["name"]) for e in mech["entities"] for x in e["ports"]]
    gs = [(e["entity"], x["name"]) for e in mech["entities"] for x in e["signals"]]
    ep = [(x["entity"], x["name"]) for x in ref["ports"]]
    es = [(x["entity"], x["name"]) for x in ref["signals"]]
    assert gp == ep and gs == es, f"{stem}: closed set drifted from {REF_DIR}"
    ok += 1
print(f"closed set: {ok}/18 modules match {REF_DIR}/ exactly, in order")
print(f"total elements: {sum(len(json.loads((REF_DIR/f'{s}.json').read_text())['ports']) + len(json.loads((REF_DIR/f'{s}.json').read_text())['signals']) for s,_ in modules)}")

18 modules | 28 record types (bus_req_t=True, bus_rsp_t=True)
closed set: 18/18 modules match parsed_tuning18/ exactly, in order
total elements: 1689


## 3. The v7 prompt

In [5]:
P = prompts_parse_v3
print("prompt   :", P.sha(P.PARSE_ELEMENTS_V3_SYSTEM), "|", len(P.PARSE_ELEMENTS_V3_SYSTEM), "chars")
print("v3 was   : 8e7e3fb0e517685d-era text; any sha change here is the four fixes")
print("edge types:", len(P.EDGE_TYPES))
print("  ", P.EDGE_TYPES)
print()
# the four changes, quoted from the prompt itself
s = P.PARSE_ELEMENTS_V3_SYSTEM
for probe, label in ((" SEQUENCES ", "SEQUENCES + RESETS"),
                     ("`guard` is REQUIRED", "the guard field"),
                     ("An input never EXPORTS", "EXPORTS direction")):
    i = s.index(probe.strip()) if probe.strip() in s else -1
    print(f"--- {label} ---")
    print("   " + s[i:i+330].replace("\n", "\n   ")[:330])
    print()
P.audit_against_corpus()
print("standing-rule audit: no corpus identifier in the prompt")

prompt   : d2569ebb9165 | 25732 chars
v3 was   : 8e7e3fb0e517685d-era text; any sha change here is the four fixes
edge types: 14
   ('SOURCES', 'CARRIES', 'DERIVES_FROM', 'CAPTURES', 'REFLECTS', 'GATES', 'SELECTS', 'CONSTRAINS', 'OVERRIDES', 'GOVERNED_BY', 'SEQUENCES', 'RESETS', 'EXPORTS', 'ISOLATED')

--- SEQUENCES + RESETS ---
   SEQUENCES
     LHS_SEQ             what appears on the right of this assignment? -> CAPTURES
     LHS_COMB            what appears on the right of this assignment? -> DERIVES_FROM, or
                         REFLECTS where every arm is a literal
     RHS                 which element is being assigned? -> SOURCES, or CARRIES wh

--- the guard field ---
   `guard` is REQUIRED on every GATES, SELECTS, CONSTRAINS, OVERRIDES, RESETS and
   GOVERNED_BY edge, and must be omitted on every other type. Quote the guard as it appears in the source, close to
   verbatim and at most 25 words: the branch test, the case choice, the enable term. Write
   `"always"` only wh

## 4. Run the parse

Cached per module — a module already in `parsed_v7_tuning18/` is free. `overwrite=True`
forces re-annotation of everything.

**Cost.** The v3 parse was 30 calls over 18 modules, the heavy ones emitting ~11.5 k output
tokens each; `neorv32_bus` alone is 10 of them. At `gpt-5-mini` that is well under **$1**.
Wall clock is a few minutes at 6 workers.

`STAGE` guards it so a run-all cannot fire the API by accident.

In [6]:
# =============================================================================
#  STAGE = 0  -> no API calls. Prints the plan only.
#  STAGE = 1  -> one module (set MOD). ~1-2 calls.
#  STAGE = 2  -> all 18 modules. ~30 calls, well under $1, cached and resumable.
# =============================================================================
STAGE = 2
MOD   = "neorv32_imem"

V7_DIR.mkdir(exist_ok=True)
todo = [m for m in modules if m[0] == MOD] if STAGE == 1 else modules
done = {p.stem for p in V7_DIR.glob("*.json")}
pending = [m for m in todo if m[0] not in done]

if STAGE == 0:
    print("STAGE = 0 : no API calls will be made.")
    print(f"  would annotate {len(pending)} of {len(todo)} module(s); "
          f"{len(todo)-len(pending)} already on disk")
    for s, _ in pending:
        print("   ", s)
else:
    parsed_annot = parse_v3.annotate_all_parallel(
        client.complete, todo, records=RECORDS, workers=PARSE_WORKERS)
    print()
    print("cost:", json.dumps(client.cost()))

standing-rule audit: no corpus identifier in either prompt
18 module(s) to annotate, 30 call(s), 6 workers

  [ 1/30] neorv32_boot_rom/neorv32_boot_rom batch 0: 21 of 21 annotated
  [ 2/30] neorv32_bus/neorv32_bus_reg batch 0: 36 of 36 annotated
  [ 3/30] neorv32_bus/neorv32_bus_switch batch 0: 62 of 62 annotated
  [ 4/30] neorv32_bus/neorv32_bus_io_switch batch 0: 128 of 128 annotated
  [ 5/30] neorv32_bus/neorv32_bus_io_switch batch 1: 129 of 128 annotated
  [ 6/30] neorv32_bus/neorv32_bus_gateway batch 0: 100 of 100 annotated
  [ 7/30] neorv32_bus/neorv32_bus_io_switch batch 3: 128 of 128 annotated
  [ 8/30] neorv32_bus/neorv32_bus_io_switch batch 2: 128 of 128 annotated
  [ 9/30] neorv32_bus/neorv32_bus_io_switch batch 4: 70 of 70 annotated
  [10/30] neorv32_bus/neorv32_bus_amo_rvs batch 0: 39 of 39 annotated
  [11/30] neorv32_bus/neorv32_bus_amo_rmw batch 0: 51 of 51 annotated
  [12/30] neorv32_cache/neorv32_cache batch 0: 64 of 64 annotated
  [13/30] neorv32_cache/neorv32_cache_m

## 5. The gate — read this before the parse is used for anything

`report()` prints the violation rate, the edge mix, and the three coverage numbers. The two
new ones are the direct read-out of the v7 changes.

| number | what it means | v3 baseline |
|---|---|---|
| violations | elements using verdict/hedging words | should be ~0 |
| governing coverage | elements in a condition that carry a control edge | **51%** |
| **guard coverage** | governing edges that record their condition | **n/a — the field did not exist** |
| **ISOLATED contradicted** | `ISOLATED` claims on elements the source uses | v3: 6 flagged by the verifier |

In [7]:
parse_v3.report(V7_DIR)

   parse version: v6/d2569ebb9165/14
module                      elems    bad    bad%  isolated
neorv32_boot_rom               21      0    0.0%        10
neorv32_bus                   870      0    0.0%         5
neorv32_cache                  86      0    0.0%         0
neorv32_cpu                   115      1    0.9%         0
neorv32_cpu_cp_cfu             29      0    0.0%         3
neorv32_cpu_cp_muldiv          70      0    0.0%        34
neorv32_cpu_pmp                62      1    1.6%        30
neorv32_debug_dtm              42      0    0.0%         5
neorv32_fifo                   27      1    3.7%         0
neorv32_hwspinlock             21      0    0.0%        10
neorv32_imem                   27      0    0.0%         8
neorv32_package                 0      0    0.0%         0
neorv32_spi                    66      3    4.5%         9
neorv32_sys                    17      0    0.0%         0
neorv32_trng                   56      1    1.8%         9
neorv32_twi        

In [8]:
# redundant with report() above, which now runs all of these. Kept
# because an explicit rtl_dir is still the clearest way to re-run one of them.
parse_v3.guard_coverage(V7_DIR)
parse_v3.audit_isolated(V7_DIR, rtl_dir=str(RTL_DIR))
parse_v3.governing_coverage(V7_DIR, rtl_dir=str(RTL_DIR))


   GUARD COVERAGE  276/282 = 98% of governing edges carry their guard
      rstn_i RESETS -> rden  when (rstn_i = '0')
      bus_req_i.addr SELECTS -> -  when to_integer(unsigned(bus_req_i.addr(boot_rom_size_ind
      rden GATES -> bus_rsp_o.data  when (rden = '1')
      rstn_i RESETS -> state  when (rstn_i = '0')

   ISOLATED CLAIMS  139 total, 18 contradicted by the source (13%)
      neorv32_boot_rom/bus_req_i.data  occurs 1x in the architecture
      neorv32_hwspinlock/bus_req_i  occurs 8x in the architecture
      neorv32_hwspinlock/bus_req_i.data  occurs 3x in the architecture
      neorv32_twi/bus_req_i  occurs 16x in the architecture
      neorv32_twi/ctrl  occurs 20x in the architecture
      neorv32_twi/fifo  occurs 31x in the architecture
      neorv32_twi/clk_gen  occurs 51x in the architecture
      neorv32_twi/engine  occurs 32x in the architecture
      neorv32_twi/io_con  occurs 33x in the architecture
      neorv32_uart/bus_req_i  occurs 28x in the architecture
      

(162, 338)

## 6. v3 versus v7

The point of a re-parse is not that the output changed — it will, the stage is
non-deterministic. It is whether the four targeted defects closed. Element counts are
asserted identical, so a difference here cannot be a closed-set difference.

In [9]:
def compare_v3_v7():
    def stats(d):
        n = Counter(); edges = Counter(); guards = [0, 0]
        for f in sorted(Path(d).glob("*.json")):
            rec = json.loads(f.read_text(encoding="utf-8"))
            for e in rec["ports"] + rec["signals"]:
                n["elements"] += 1
                for r in e.get("relationship") or []:
                    t = str(r.get("type", "")).upper()
                    edges[t] += 1
                    if t in parse_v3.GOVERNING_EDGES:
                        guards[0 if str(r.get("guard", "")).strip() else 1] += 1
        return n, edges, guards

    if not any(Path(V7_DIR).glob("*.json")):
        print(f"{V7_DIR}/ is empty -- run the parse cell first"); return
    print("NOTE: v3 and v7 use DIFFERENT EDGE SETS. CARRIES points the opposite\n      way, SLICES and AGGREGATES are retired, GOVERNED_BY and RESETS are new.\n      Read the per-type rows as a vocabulary change, not as a quality delta.\n")
    n3, e3, g3 = stats(V3_DIR)
    n7, e7, g7 = stats(V7_DIR)
    print(f"{'':26} {'v3':>8} {'v7':>8} {'delta':>8}")
    print(f"{'elements':26} {n3['elements']:8} {n7['elements']:8} {n7['elements']-n3['elements']:+8}")
    print(f"{'edges':26} {sum(e3.values()):8} {sum(e7.values()):8} {sum(e7.values())-sum(e3.values()):+8}")
    print()
    print(f"{'edge type':26} {'v3':>8} {'v7':>8} {'delta':>8}")
    for t in sorted(set(e3) | set(e7), key=lambda x: -(e3.get(x, 0) + e7.get(x, 0))):
        print(f"  {t:24} {e3.get(t,0):8} {e7.get(t,0):8} {e7.get(t,0)-e3.get(t,0):+8}")
    print()
    ctrl3 = sum(e3.get(t, 0) for t in parse_v3.GOVERNING_EDGES)
    ctrl4 = sum(e7.get(t, 0) for t in parse_v3.GOVERNING_EDGES)
    print(f"control edges              {ctrl3:8} {ctrl4:8} {ctrl4-ctrl3:+8}"
          f"   ({100*ctrl3/max(1,sum(e3.values())):.1f}% -> {100*ctrl4/max(1,sum(e7.values())):.1f}%)")
    print(f"SEQUENCES on reset         {e3.get('SEQUENCES',0):8} {e7.get('SEQUENCES',0):8}"
          f"   RESETS now: {e7.get('RESETS',0)}")
    print(f"governing edges with guard {'n/a':>8} {g7[0]:8}"
          f"   missing: {g7[1]}")

compare_v3_v7()

NOTE: v3 and v6 use DIFFERENT EDGE SETS. CARRIES points the opposite
      way, SLICES and AGGREGATES are retired, GOVERNED_BY and RESETS are new.
      Read the per-type rows as a vocabulary change, not as a quality delta.

                                 v3       v6    delta
elements                       1689     1689       +0
edges                          2464     2714     +250

edge type                        v3       v6    delta
  SOURCES                       646      660      +14
  EXPORTS                       430      473      +43
  DERIVES_FROM                  217      642     +425
  CARRIES                       460       43     -417
  CAPTURES                      216      218       +2
  GATES                         149      186      +37
  ISOLATED                      110      139      +29
  GOVERNED_BY                     0      209     +209
  SEQUENCES                      76       32      -44
  SELECTS                        39       37       -2
  RESETS          

## What to look at, in order

1. **Did `SEQUENCES` drop and `RESETS` appear?** If `RESETS` is near zero the edge is not
   being used and the definition needs work, not the model.
2. **Guard coverage.** This is the number the whole change exists for. An empty guard on a
   governing edge is the sweep not landing, and it is now countable.
3. **Governing coverage against the 51% baseline.** Guard coverage can be high while
   governing coverage stays low — that would mean the edges that get written are well
   described, but the sweep is still missing edges entirely. Two different failures.
4. **ISOLATED contradicted.** Should be 0. It is a mechanical check, so anything above 0 is
   the model overriding a rule the source can settle.
5. **Violation rate.** Unchanged from v3 — the banned-word gate was not touched.

### What this notebook does not do

It does not identify assets, and it must not. The parser supplies mechanism; the asset stage
supplies the verdict. If a `role` here starts describing significance, the asset stage is
reading its own rubric back from its input — which is what the banned-word gate exists to
prevent.